# P4 final observability analysis

This notebook is the cleaned P4 observability result for the final August 22 method.

- Optimized first use: 100 cycles, 1.50 C charge, 1.25 C discharge, 3.1–4.1 V, C/50 hold, 0.1 h rest.
- Optimized second use: 2.00 C charge, 1.25 C discharge, 3.0–4.1 V, C/50 hold, 0.1 h rest.
- Solver: RK23 with `rtol=1e-7`, componentwise scaled absolute tolerance, first step of 10 cycles, and no maximum-step restriction.
- Empirical perturbation: independent ±1% changes in each of five DeepSOH states.
- Sampling: one saved observation per simulated cycle, cycles 0–120.

The notebook uses the bundled, validated 11-trajectory data so the analysis and figures rerun quickly without repeating the expensive electrochemical simulations.

## Configuration and bundled inputs

In [ ]:
from collections import OrderedDict
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the deliverable folder from the working directory, so the notebook
# runs from a fresh checkout with no machine-specific path.
FINAL_ROOT = Path.cwd()
if not (FINAL_ROOT / "data").exists():
    for _base in (Path.cwd(), *Path.cwd().parents):
        _cand = _base / "deepSOH_final_august22_2026"
        if (_cand / "data").exists():
            FINAL_ROOT = _cand
            break

DATA_ROOT = FINAL_ROOT / "data" / "observability"
OUTPUT_ROOT = FINAL_ROOT / "outputs" / "observability"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TRAJECTORY_CSV = DATA_ROOT / "P4_trajectory_components.csv"
METADATA_JSON = DATA_ROOT / "P4_metadata.json"

STATE_NAMES = ("nLi", "Cp", "Cn", "delta_SEI", "delta_pl")
STATE_COLUMNS = ("nLi_mol", "Cp_Ah", "Cn_Ah", "delta_SEI_m", "delta_pl_m")
STATE_LABELS = (
    r"$n_{\mathrm{Li}}$ [mol]", r"$C_p$ [Ah]", r"$C_n$ [Ah]",
    r"$\delta_{\mathrm{SEI}}$ [m]", r"$\delta_{\mathrm{plating}}$ [m]",
)
OUTPUT_COLUMNS = ("capacity_Ah", "resistance_Ohm", "expansion_um")
OUTPUT_LABELS = ("Capacity [Ah]", "Resistance [Ohm]", "Expansion [um]")
# eSOH from an OCV fit yields three deepSOH states directly: n_Li, C_p, C_n
# (Eq. 22). Sensitivity is computed for these alongside the three scalar
# outputs, each channel carrying the same 1% relative measurement noise.
ESOH_COLUMNS = ("nLi_mol", "Cp_Ah", "Cn_Ah")
SENSITIVITY_COLUMNS = OUTPUT_COLUMNS + ESOH_COLUMNS
PERTURBATION_FRACTION = 0.01
NOISE_FRACTION = 0.01

# indices below point into SENSITIVITY_COLUMNS:
#   0 capacity, 1 resistance, 2 expansion, 3 n_Li, 4 C_p, 5 C_n; eSOH = {3,4,5}
MEASUREMENT_SETS = OrderedDict([
    ("Capacity", (0,)),
    ("Resistance", (1,)),
    ("Capacity + Resistance", (0, 1)),
    ("Expansion", (2,)),
    ("Resistance + Expansion", (1, 2)),
    ("Capacity + Resistance + Expansion", (0, 1, 2)),
    ("eSOH", (3, 4, 5)),
    ("eSOH + Resistance", (3, 4, 5, 1)),
    ("eSOH + Resistance + Expansion", (3, 4, 5, 1, 2)),
])
# (color, matplotlib linestyle, marker) per set. The three sets carried over
# from the original Figure 7 keep their colors; the eSOH sets use colors not
# used by any original set.
STYLES = OrderedDict([
    ("Capacity", ("#0072B2", "-", "o")),
    ("Resistance", ("#D55E00", "--", "x")),
    ("Capacity + Resistance", ("#009E73", "-.", "s")),
    ("Expansion", ("#CC79A7", ":", "v")),
    ("Resistance + Expansion", ("#E69F00", "--", "D")),
    ("Capacity + Resistance + Expansion", ("#000000", (0, (3, 1, 1, 1)), "P")),
    ("eSOH", ("#9467BD", "-", "^")),
    ("eSOH + Resistance", ("#8C564B", "--", "P")),
    ("eSOH + Resistance + Expansion", ("#000000", "-.", "*")),
])
# sets shown in the prediction-variance figure (Capacity + Resistance is
# computed for the sigma_min figure but not shown here)
IOPT_PLOT_SETS = ("Capacity", "Resistance", "eSOH",
                  "Capacity + Resistance", "Resistance + Expansion",
                  "eSOH + Resistance", "eSOH + Resistance + Expansion")

metadata = json.loads(METADATA_JSON.read_text(encoding="utf-8"))
display(pd.DataFrame([
    {
        "model": metadata["model"],
        "solver": metadata["solver"]["method"],
        "rtol": metadata["solver"]["rtol"],
        "perturbation": metadata["perturbation_fraction"],
        "saved_cycles": metadata["common_number_of_cycle_samples"],
        "upper_voltage_V": metadata["protocol"]["upper_V"],
    }
]))

## Validate the 11 trajectories

In [ ]:
trajectories = pd.read_csv(TRAJECTORY_CSV)
expected_labels = ("nominal",) + tuple(
    f"{state}_{suffix}"
    for state in STATE_NAMES
    for suffix in ("plus", "minus")
)

found_labels = tuple(trajectories["trajectory"].drop_duplicates())
if set(found_labels) != set(expected_labels):
    raise AssertionError(f"Unexpected trajectories: {found_labels}")

cycle_grid = None
blocks = {}
for label in expected_labels:
    block = trajectories.loc[trajectories["trajectory"] == label].sort_values("cycle")
    cycles = block["cycle"].to_numpy(dtype=float)
    if cycle_grid is None:
        cycle_grid = cycles
    else:
        np.testing.assert_allclose(cycle_grid, cycles, rtol=0, atol=1e-10)
    blocks[label] = block

validation = pd.DataFrame([
    {
        "trajectory_count": len(blocks),
        "first_cycle": cycle_grid[0],
        "last_cycle": cycle_grid[-1],
        "observations_per_trajectory": len(cycle_grid),
        "off_target_restart_error_max": max(
            row["maximum_scaled_initial_error"] for row in metadata["trajectories"]
        ),
    }
])
display(validation)

## The 11 trajectories

One nominal trajectory and a plus/minus perturbation for each of
$n_{\mathrm{Li}}$, $C_p$, $C_n$, $\delta_{\mathrm{SEI}}$, and
$\delta_{\mathrm{plating}}$.

In [ ]:
trajectory_styles = {
    "nominal": ("#000000", "-"),
    "nLi_plus": ("#0072B2", "-"), "nLi_minus": ("#0072B2", "--"),
    "Cp_plus": ("#D55E00", "-"), "Cp_minus": ("#D55E00", "--"),
    "Cn_plus": ("#009E73", "-"), "Cn_minus": ("#009E73", "--"),
    "delta_SEI_plus": ("#E69F00", "-"), "delta_SEI_minus": ("#E69F00", "--"),
    "delta_pl_plus": ("#CC79A7", "-"), "delta_pl_minus": ("#CC79A7", "--"),
}
panels = tuple(zip(OUTPUT_COLUMNS, OUTPUT_LABELS)) + tuple(zip(STATE_COLUMNS, STATE_LABELS))
fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True, constrained_layout=True)
flat = axes.ravel()
for label in expected_labels:
    block = blocks[label]
    color, linestyle = trajectory_styles[label]
    for index, (column, ylabel) in enumerate(panels):
        flat[index].plot(
            block["cycle"], block[column], color=color, linestyle=linestyle,
            linewidth=1.35, label=label,
        )
        flat[index].set_ylabel(ylabel)
        flat[index].grid(True, alpha=0.25)
for axis in flat[:8]:
    axis.set_xlabel("Second-life cycle")
flat[8].axis("off")
handles, labels = flat[0].get_legend_handles_labels()
flat[8].legend(handles, labels, loc="upper left", frameon=False, fontsize=8)
fig.suptitle("P4 final tight-RK23 observability trajectories (second use)", fontsize=14)
trajectory_png = OUTPUT_ROOT / "P4_observability_11_trajectories.png"
fig.savefig(trajectory_png, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

## Empirical observability Gramian and I-optimality

Outputs are normalized by the nominal cycle-zero value. The cumulative
Gramian is integrated by the trapezoidal rule. I-optimality always predicts
future Capacity + Resistance. Lower I-optimality is better; higher
$\sigma_{\min}(W)$ and lower condition number are better.

In [ ]:
def numerical_rank(singular_values, n_rows, n_cols):
    if singular_values.size == 0 or singular_values[0] == 0:
        return 0
    tolerance = max(n_rows, n_cols) * np.finfo(float).eps * singular_values[0]
    return int(np.count_nonzero(singular_values > tolerance))


reference = blocks["nominal"].loc[:, SENSITIVITY_COLUMNS].iloc[0].to_numpy(dtype=float)
raw_sensitivity = np.empty((len(cycle_grid), len(SENSITIVITY_COLUMNS), 5), dtype=float)
for state_index, state in enumerate(STATE_NAMES):
    plus = blocks[f"{state}_plus"].loc[:, SENSITIVITY_COLUMNS].to_numpy(dtype=float)
    minus = blocks[f"{state}_minus"].loc[:, SENSITIVITY_COLUMNS].to_numpy(dtype=float)
    raw_sensitivity[:, :, state_index] = (
        plus - minus
    ) / (2 * PERTURBATION_FRACTION)
normalized = raw_sensitivity / reference[None, :, None]
whitened = normalized / NOISE_FRACTION


def gramian_rows():
    rows = []
    for measurement_name, indices in MEASUREMENT_SETS.items():
        selected = normalized[:, indices, :]
        W = np.zeros((5, 5), dtype=float)
        # plain per-instant sum, matching Eq. (22') in the paper and the
        # stacked-row Gramian used by the prediction-variance code below
        for index, cycle in enumerate(cycle_grid):
            W += selected[index].T @ selected[index]
            singular = np.linalg.svd(W, compute_uv=False)
            rank = numerical_rank(singular, 5, 5)
            sigma_min = float(singular[-1])
            condition = (
                float(singular[0] / sigma_min)
                if rank == 5 and sigma_min > 0 else np.inf
            )
            rows.extend([
                {"metric": "sigma_min_W", "measurement_set": measurement_name,
                 "horizon_cycle": cycle, "service_N": np.nan, "rank": rank,
                 "value": sigma_min},
                {"metric": "condition_W", "measurement_set": measurement_name,
                 "horizon_cycle": cycle, "service_N": np.nan, "rank": rank,
                 "value": condition},
            ])
    return rows


def ioptimality_rows():
    rows = []
    future_indices = (0, 1)
    modes = (("N10", 10), ("N25", 25), ("remaining", None))
    for measurement_name, measurement_indices in MEASUREMENT_SETS.items():
        for horizon_index in range(1, len(cycle_grid)):
            diagnostic = whitened[:horizon_index, measurement_indices, :].reshape(-1, 5)
            _, singular, vt = np.linalg.svd(diagnostic, full_matrices=False)
            rank = numerical_rank(singular, *diagnostic.shape)
            for mode, fixed_n in modes:
                service_n = len(cycle_grid) - horizon_index if fixed_n is None else fixed_n
                if service_n < 1 or horizon_index + service_n > len(cycle_grid):
                    continue
                value = np.nan
                if rank == 5:
                    future = whitened[
                        horizon_index:horizon_index + service_n, future_indices, :
                    ].reshape(-1, 5)
                    transformed = (future @ vt.T) / singular[None, :]
                    value = float(
                        np.sum(transformed * transformed) / (service_n * len(future_indices))
                    )
                rows.append({
                    "metric": f"Ioptimality_{mode}",
                    "measurement_set": measurement_name,
                    "horizon_cycle": cycle_grid[horizon_index],
                    "service_N": service_n,
                    "rank": rank,
                    "value": value,
                })
    return rows


metric_curves = pd.DataFrame(gramian_rows() + ioptimality_rows())
metric_csv = OUTPUT_ROOT / "P4_observability_metric_curves.csv"
metric_curves.to_csv(metric_csv, index=False)

final_gramian = metric_curves.loc[
    (metric_curves["horizon_cycle"] == cycle_grid[-1])
    & metric_curves["metric"].isin(["sigma_min_W", "condition_W"])
].pivot(index="measurement_set", columns="metric", values="value")
display(final_gramian)

## Final observability figure

In [ ]:
plot_specs = (
    ("sigma_min_W", r"$\sigma_{\min}(W)$: Capacity + Resistance",
     r"$\sigma_{\min}(W)$ (higher is better)", "Last cycle included in W", True),
    ("Ioptimality_N10", "I-optimality: next 10 cycles",
     "Mean normalized prediction variance", "Diagnostic horizon H [cycles]", False),
    ("Ioptimality_remaining", "I-optimality: all remaining cycles",
     "Mean normalized prediction variance", "Diagnostic horizon H [cycles]", False),
)
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
flat = axes.ravel()
for axis, (metric, title, ylabel, xlabel, cr_only) in zip(flat[:3], plot_specs):
    names = ("Capacity + Resistance",) if cr_only else IOPT_PLOT_SETS
    for measurement_name in names:
        subset = metric_curves.loc[
            (metric_curves["metric"] == metric)
            & (metric_curves["measurement_set"] == measurement_name)
        ].sort_values("horizon_cycle")
        subset = subset.loc[np.isfinite(subset["value"]) & (subset["value"] > 0)]
        color, linestyle, _marker = STYLES[measurement_name]
        axis.plot(subset["horizon_cycle"], subset["value"], color=color,
                  linestyle=linestyle, linewidth=1.8)
    axis.set_yscale("log")
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    axis.grid(True, which="both", alpha=0.25)
legend_axis = flat[3]
legend_axis.axis("off")
handles = [
    Line2D([0], [0], color=STYLES[name][0], linestyle=STYLES[name][1], linewidth=2, label=name)
    for name in IOPT_PLOT_SETS
]
legend_axis.legend(handles=handles, title="Measurement combination (I-optimality panels)",
                   loc="upper left", frameon=False)
note = chr(10).join([
    "sigma_min panel: Capacity + Resistance only",
    "I-optimality target: Capacity + Resistance",
    "N = 10 or all remaining cycles",
    "One observation per cycle; cycle-zero normalization, 1% noise",
])
legend_axis.text(0, 0.34, note, transform=legend_axis.transAxes, va="top")
fig.suptitle("P4 final observability metrics — tight RK23, ±1% perturbations", fontsize=15)
metrics_png = OUTPUT_ROOT / "P4_observability_all_metrics.png"
fig.savefig(metrics_png, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

## Export plotting data to MATLAB (.mat)

In [ ]:
from scipy.io import savemat

MAT_ROOT = FINAL_ROOT / "matlab"
MAT_ROOT.mkdir(parents=True, exist_ok=True)

def _hex2rgb(h):
    h = h.lstrip("#")
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

def _mls(ls):
    return ls if isinstance(ls, str) and ls in ("-", "--", "-.", ":") else "-"

_all_cols = list(OUTPUT_COLUMNS) + list(STATE_COLUMNS)
_labels = list(expected_labels)
_ncyc = len(cycle_grid)
mat = {
    "cycle": np.asarray(cycle_grid, dtype=float),
    "labels": np.array(_labels, dtype=object),
    "traj_colors": np.array([_hex2rgb(trajectory_styles[l][0]) for l in _labels], dtype=float),
    "traj_linestyles": np.array([_mls(trajectory_styles[l][1]) for l in _labels], dtype=object),
    "output_columns": np.array(list(OUTPUT_COLUMNS), dtype=object),
    "state_columns": np.array(list(STATE_COLUMNS), dtype=object),
    "meas_sets": np.array(list(MEASUREMENT_SETS), dtype=object),
    "meas_colors": np.array([_hex2rgb(STYLES[m][0]) for m in MEASUREMENT_SETS], dtype=float),
    "meas_linestyles": np.array([_mls(STYLES[m][1]) for m in MEASUREMENT_SETS], dtype=object),
    "iopt_sets": np.array(list(IOPT_PLOT_SETS), dtype=object),
    "iopt_colors": np.array([_hex2rgb(STYLES[m][0]) for m in IOPT_PLOT_SETS], dtype=float),
    "iopt_linestyles": np.array([_mls(STYLES[m][1]) for m in IOPT_PLOT_SETS], dtype=object),
    "m_metric": np.array(metric_curves["metric"].astype(str).tolist(), dtype=object),
    "m_set": np.array(metric_curves["measurement_set"].astype(str).tolist(), dtype=object),
    "m_cycle": metric_curves["horizon_cycle"].to_numpy(dtype=float),
    "m_value": metric_curves["value"].to_numpy(dtype=float),
}
for col in _all_cols:
    arr = np.zeros((_ncyc, len(_labels)))
    for j, lab in enumerate(_labels):
        arr[:, j] = blocks[lab].sort_values("cycle")[col].to_numpy(dtype=float)
    mat[col] = arr
savemat(str(MAT_ROOT / "P4_01_observability.mat"), mat, do_compression=True)
print("saved", MAT_ROOT / "P4_01_observability.mat")